# Sample agent and teammate handoff

Run this notebook independently to inspect the shareable [agent template](agent_template/README.md), test its legal-move interface, play native games, and build a ZIP for teammates. The weighted scoring demo is ready to run; its JSON weights are hand-set, not trained.

Use the form controls in Colab, or edit their variables in Jupyter. Run from top to bottom. A separate [RL notebook](reinforcement_learning.ipynb) trains the Transformer/MCTS baseline, and the [team arena](arena.ipynb) compares any compatible agents.

**Colab: choose Runtime → Run all.** Setup clones the public [project repository](https://github.com/le21-j/pokemon-tcg-fpga) and installs dependencies automatically.

In [ ]:
# @title 1. Automatic setup {display-mode: "form"}
# @markdown Run All clones the public project and installs dependencies automatically in Colab. Optional overrides can stay at their defaults.
REPOSITORY_PATH = "" # @param {type:"string"}
MOUNT_GOOGLE_DRIVE = False # @param {type:"boolean"}
INSTALL_LOCALLY = False # @param {type:"boolean"}

"""Standard-library Colab setup for the public Pokémon TCG project repository."""

import os
from pathlib import Path
import subprocess
import sys
import tempfile


PUBLIC_REPOSITORY = "https://github.com/le21-j/pokemon-tcg-fpga.git"


def running_in_colab():
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
        return True
    try:
        import google.colab
        return True
    except ImportError:
        return False


def _project_ready(root):
    return all((root / name).is_file() for name in (
        "project/arena.py", "project/sample_sdk.py", "project/agent/model.py",
        "project/rl_baseline/train.py", "project/agent_template/agent.py",
        "project/requirements-colab.txt"))


def setup_project(repository_path="", *, repository_url=PUBLIC_REPOSITORY, repository_ref="main",
                  in_colab=None, workspace_parent=None, install_locally=False):
    in_colab = running_in_colab() if in_colab is None else in_colab
    if repository_path.strip():
        root = Path(repository_path).expanduser().resolve()
        if not _project_ready(root):
            raise ValueError("REPOSITORY_PATH must contain the Pokémon TCG project")
    elif not in_colab:
        root = next((p for p in [Path.cwd(), *Path.cwd().parents] if _project_ready(p)), None)
        if root is None:
            raise ValueError("Run inside the project checkout or set REPOSITORY_PATH")
    else:
        parent = Path(workspace_parent) if workspace_parent is not None else (
            Path("/content") if Path("/content").is_dir() else Path.cwd())
        parent.mkdir(parents=True, exist_ok=True)
        name = "pokemon-tcg-fpga"
        root = parent / name
        retry = 0
        while root.exists() and not _project_ready(root):
            retry += 1
            root = parent / f"{name}-{retry}"
        if not root.exists():
            with tempfile.TemporaryDirectory(prefix="tcg-clone-", dir=parent) as temporary:
                checkout = Path(temporary) / "checkout"
                subprocess.run(["git", "clone", "--depth", "1", "--branch", repository_ref,
                                repository_url, str(checkout)],
                               env=dict(os.environ, GIT_TERMINAL_PROMPT="0"), check=True)
                if not _project_ready(checkout):
                    raise RuntimeError("The downloaded repository is missing project files")
                checkout.rename(root)
            print("Cloned the public Pokémon TCG project")
    if in_colab or install_locally:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                               "-r", str(root / "project/requirements-colab.txt")])
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print("Project ready:", root)
    return root


if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = setup_project(REPOSITORY_PATH, install_locally=INSTALL_LOCALLY)

# --- End automatic project setup ---

import json
import uuid
import pandas as pd
from IPython.display import display, Code
DOWNLOAD_RUNTIME_IF_MISSING = True

from project.collect_matches import ENGINE_ROOT, download_runtime, load_runtime, runtime_identity, choose_random
from project.agent.native_search import NativeSearch
from project.arena import Agent, load_deck_pool, priority_action, run_series, validate_action
from project.script_agent import ScriptAgent

if not (ENGINE_ROOT / "cg/game.py").exists():
    if not DOWNLOAD_RUNTIME_IF_MISSING:
        raise FileNotFoundError("Enable runtime download or prepare the native runtime")
    download_runtime(ENGINE_ROOT)
game_api, sim = load_runtime(ENGINE_ROOT)
search_api = NativeSearch(sim.Battle.lib)
DECK_POOL = load_deck_pool()
print("Workspace:", ROOT)
print("Python:", sys.executable)

In [ ]:
# @title 2. Select the sample folder and matchup {display-mode: "form"}
AGENT_DIRECTORY = "project/agent_template" # @param {type:"string"}
AGENT_NAME = "Template demo" # @param {type:"string"}
OPPONENT = "Random" # @param ["Random", "Priority"]
OWN_DECK = "sample-water" # @param ["random", "sample-water", "mega-lucario", "mega-abomasnow", "dragapult"]
OPPONENT_DECK = "random" # @param ["random", "sample-water", "mega-lucario", "mega-abomasnow", "dragapult"]
SERIES_SEED = 7 # @param {type:"integer"}

folder = Path(AGENT_DIRECTORY).expanduser()
folder = folder.resolve() if folder.is_absolute() else (ROOT / folder).resolve()
if not (folder / "agent.py").is_file():
    raise ValueError("Choose the template folder or a copy containing agent.py")
for path in sorted(folder.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts and path.name != ".DS_Store":
        print(path.relative_to(folder))

`agent.py` loads the model once, `model.py` defines inference, and `weights/` holds model assets. Teammates replace these parts together, keeping training-time preprocessing consistent. The optional `start_game(own_deck)` hook resets history each game.

In [ ]:
# @title 3. Inspect the entry point and run its small interface check {display-mode: "form"}
display(Code((folder / "agent.py").read_text(), language="python"))
check = folder / "check_agent.py"
if check.is_file():
    completed = subprocess.run([sys.executable, str(check)], cwd=folder,
                               check=True, capture_output=True, text=True)
    print(completed.stdout)
else:
    print("No synthetic check provided; the next cells test real native observations")

In [ ]:
# @title 4. Ask the script for a move in a real game {display-mode: "form"}
import random

sample_deck = DECK_POOL["sample-water"]
actor = ScriptAgent(folder)
try:
    actor.start_game(sample_deck)
    observation, start = game_api.battle_start(sample_deck, sample_deck)
    if start.errorPlayer >= 0 or observation is None:
        raise ValueError(f"Native deck error {start.errorType}")
    display(pd.DataFrame([{"option index": i, **option}
                          for i, option in enumerate(observation["select"]["option"])]))
    action = actor(observation, random.Random(SERIES_SEED))
    validate_action(observation, action)
    print("Permitted count:", observation["select"]["minCount"], "to", observation["select"]["maxCount"])
    print("Returned legal option indices:", action)
finally:
    actor.close()
    if sim.Battle.battle_ptr:
        game_api.battle_finish()
        sim.Battle.battle_ptr = None

In [ ]:
# @title 5. Play a first-to-three series {display-mode: "form"}
actor = ScriptAgent(folder)
a = Agent(AGENT_NAME, actor, OWN_DECK)
b = Agent(f"{OPPONENT} baseline", choose_random if OPPONENT == "Random" else priority_action, OPPONENT_DECK)
try:
    series = run_series(game_api, sim, a, b, DECK_POOL, seed=SERIES_SEED)
finally:
    actor.close()
assert sim.Battle.battle_ptr is None
assert series["completed"] and max(series["wins"]) == 3
print("Winner:", series["winner"], "Score:", series["wins"])
display(pd.DataFrame([{"game": g["game"], "winner": g["winner"], "score": g["score"],
                       "decisions": g["decisions"], "seconds": g["seconds"]} for g in series["games"]]))

In [ ]:
# @title 6. Package the folder for teammates {display-mode: "form"}
# @markdown The ZIP includes agent.py, model.py, requirements, README, and weights. It excludes Python caches. Enable the download checkbox when using Colab.
DOWNLOAD_ZIP = False # @param {type:"boolean"}

from zipfile import ZipFile, ZIP_DEFLATED
run_directory = ROOT / "project/results/sample-agent" / uuid.uuid4().hex[:12]
run_directory.mkdir(parents=True, exist_ok=False)
archive = run_directory / f"{folder.name}.zip"
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for path in sorted(folder.rglob("*")):
        if path.is_file() and "__pycache__" not in path.parts and path.name != ".DS_Store":
            bundle.write(path, str(Path(folder.name) / path.relative_to(folder)))
series["runtime"] = runtime_identity(ENGINE_ROOT)
series["competitors"] = [actor.identity(), {"source": OPPONENT}]
(run_directory / "series.json").write_text(json.dumps(series, indent=2) + "\n")
print("Share this ZIP:", archive)
print("Saved match report:", run_directory / "series.json")
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(archive))

Teammates select **Python script** in `arena.ipynb`, point to their unzipped folder, and leave the default `agent`/`observation` entry point. They install their model's requirements in the chosen Python environment. Send the inference code, model definition, weights, vocabulary/config files, and a model-version name together.

The synthetic check covers only its supplied fixtures; full game testing exercises real observations and selections. The arena checks option indices and permitted counts before calling the native engine. Native deals/shuffles are fresh but have no exposed seed.

[Agent template instructions](agent_template/README.md) · [CABT game API](https://matsuoinstitute.github.io/cabt/)